# Dimensionality reduction for mass spectrometry feature tables

Scenario: You are analyzing an untargeted LC-MS metabolomics experiment with three biological conditions:

- **Control:** untreated cell extracts.
- **Stress:** cells exposed to oxidative stress.
- **Rescue:** stressed cells treated with a rescue compound.

The experiment also includes pooled QC injections and procedural blanks. Your task is to determine what separates the samples and whether the apparent structure reflects biology, batch, drift, extraction, or contamination.


## How to use this notebook

Most cells are complete and can be run directly. Hands-on work appears in two forms:

1. **Parameter edits:** change values such as `COLOR_BY`, `DISTANCE_METRIC`, or `PERPLEXITY`.
2. **Interpretation prompts:** answer the questions in markdown or discuss them with your group.

Cells marked **Intermediate challenge** are optional. They are useful for learners who already have some Python experience.


## Import Python packages

In [ ]:
# Import NumPy for numerical arrays and random-number generation.
import numpy as np

# Import pandas for tabular data structures such as DataFrames.
import pandas as pd

# Import matplotlib for plotting.
import matplotlib.pyplot as plt

# Import PCA from scikit-learn for principal component analysis.
from sklearn.decomposition import PCA

# Import classical MDS from scikit-learn for distance-matrix ordination.
from sklearn.manifold import ClassicalMDS

# Import StandardScaler from scikit-learn for z-score scaling.
from sklearn.preprocessing import StandardScaler

# Import silhouette_score to quantify how well groups separate in an embedding.
from sklearn.metrics import silhouette_score

# Import pairwise distance helpers from SciPy for PCoA.
from scipy.spatial.distance import pdist, squareform

# Import stats from SciPy for simple correlations.
from scipy import stats

# Configure pandas to display more columns when showing feature metadata tables.
pd.set_option("display.max_columns", 30)

# Configure pandas to display a reasonable number of rows in result tables.
pd.set_option("display.max_rows", 20)

# Set a fixed random seed so that the simulated data are exactly reproducible.
RANDOM_SEED = 7

# Create a NumPy random-number generator object from the fixed seed.
rng = np.random.default_rng(RANDOM_SEED)

## Simulate a realistic LC-MS feature table

Here we simulate a feature table, which is the final object that mass spectrometrists usually analyze after peak picking and alignment:

- one row per sample;
- one column per LC-MS feature;
- intensities as positive numbers, with zeros representing non-detected features;
- a separate sample metadata table;
- a separate feature metadata table.

In [ ]:
# Define the number of biological replicates for each experimental condition.
n_replicates_per_condition = 18

# Define the biological conditions in the simulated experiment.
conditions = ["Control", "Stress", "Rescue"]

# Create an empty list that will hold experimental sample records before run-order scheduling.
experimental_pool = []

# Loop over each biological condition.
for condition in conditions:
    # Loop over replicate numbers from 1 to n_replicates_per_condition.
    for replicate in range(1, n_replicates_per_condition + 1):
        # Create a sample identifier such as Control_01 or Stress_14.
        sample_id = f"{condition}_{replicate:02d}"
        # Assign extraction protocols in a balanced way within each condition.
        extraction_protocol = "Protocol_A" if replicate % 2 == 1 else "Protocol_B"
        # Store the sample-level information in a dictionary.
        experimental_pool.append(
            {
                "sample_id": sample_id,
                "sample_type": "Experimental",
                "condition": condition,
                "extraction_protocol": extraction_protocol,
            }
        )

# Shuffle the experimental samples to mimic randomized acquisition order.
rng.shuffle(experimental_pool)

# Define injection positions that will be procedural blanks.
blank_positions = {1, 2, 23, 44, 67, 68}

# Define injection positions that will be pooled QC samples.
qc_positions = {8, 16, 24, 32, 40, 48, 56, 64}

# Define the total number of injections in the simulated run.
n_injections = len(experimental_pool) + len(blank_positions) + len(qc_positions)

# Create an empty list that will hold the run-ordered sample metadata records.
run_records = []

# Create counters for naming QC and blank samples.
qc_counter = 0
blank_counter = 0

# Loop over the injection order from 1 to n_injections.
for injection_order in range(1, n_injections + 1):
    # Check whether the current injection is a procedural blank.
    if injection_order in blank_positions:
        # Increase the blank counter by one.
        blank_counter += 1
        # Create a metadata record for this blank injection.
        record = {
            "sample_id": f"Blank_{blank_counter:02d}",
            "sample_type": "Blank",
            "condition": "Blank",
            "extraction_protocol": "Blank",
        }
    # Check whether the current injection is a pooled QC sample.
    elif injection_order in qc_positions:
        # Increase the QC counter by one.
        qc_counter += 1
        # Create a metadata record for this pooled QC injection.
        record = {
            "sample_id": f"QC_{qc_counter:02d}",
            "sample_type": "QC",
            "condition": "QC_pool",
            "extraction_protocol": "Pooled_QC",
        }
    # Otherwise, the injection is an experimental biological sample.
    else:
        # Pop one randomized experimental sample from the pool.
        record = experimental_pool.pop()
    # Store the injection order in the metadata record.
    record["injection_order"] = injection_order
    # Assign two LC batches based on the first and second halves of the run.
    record["lc_batch"] = (
        "LC_Batch_1" if injection_order <= n_injections / 2 else "LC_Batch_2"
    )
    # Assign two columns to make the batch effect feel realistic.
    record["column"] = "Column_A" if injection_order <= n_injections / 2 else "Column_B"
    # Add the complete record to the run metadata list.
    run_records.append(record)

# Convert the list of dictionaries into a pandas DataFrame.
sample_metadata = pd.DataFrame(run_records)

# Use the sample identifiers as the DataFrame index for easy alignment with the feature table.
sample_metadata = sample_metadata.set_index("sample_id")

# Show the first few rows of the sample metadata table.
sample_metadata.head(10)

### Interpretation prompt

Look at the metadata above. Why are pooled QCs and procedural blanks useful in a dimensionality reduction exercise?

In [ ]:
# Define the number of LC-MS features in the simulated dataset.
n_features = 360

# Define how many features belong to each simulated effect category.
driver_counts = {
    "biology_stress_up": 45,
    "biology_rescue_up": 45,
    "lc_batch_shift": 35,
    "injection_drift": 25,
    "extraction_protocol_shift": 20,
    "blank_contaminant": 25,
    "background": 165,
}

# Create a list with one driver label per feature.
driver_types = []

# Loop over each driver type and its requested count.
for driver_type, count in driver_counts.items():
    # Extend the list by repeating the driver label count times.
    driver_types.extend([driver_type] * count)

# Convert the driver-type list to a NumPy array for easier indexing.
driver_types = np.array(driver_types)

# Randomly shuffle feature driver labels so they are not ordered by feature ID.
rng.shuffle(driver_types)

# Define plausible compound classes for untargeted LC--MS features.
compound_classes = [
    "Amino acid / amine",
    "Organic acid",
    "Acylcarnitine",
    "Glycerophospholipid",
    "Sphingolipid",
    "Nucleotide",
    "Xenobiotic / contaminant",
    "Unknown",
]

# Define possible adduct labels for positive-mode LC-MS data.
adducts = ["[M+H]+", "[M+Na]+", "[M+K]+", "[M+NH4]+", "[M-H2O+H]+"]

# Define possible annotation confidence labels.
annotation_levels = ["MSI level 2", "MSI level 3", "MSI level 4 / unknown"]

# Create an empty list to hold feature metadata records.
feature_records = []


# Define a helper function to sample m/z and retention time by compound class.
def sample_mz_rt(compound_class, rng):
    # Low-mass polar metabolites tend to elute early.
    if compound_class in ["Amino acid / amine", "Organic acid"]:
        # Draw m/z values from a low-to-mid mass range.
        mz = rng.uniform(80, 320)
        # Draw retention times from an early-eluting range.
        rt = rng.uniform(0.4, 4.5)
    # Acylcarnitines are often mid-mass and mid-retention in reversed-phase LC.
    elif compound_class == "Acylcarnitine":
        # Draw m/z values from a mid-mass range.
        mz = rng.uniform(250, 550)
        # Draw retention times from a mid-gradient range.
        rt = rng.uniform(3.0, 8.5)
    # Lipids tend to be higher mass and later eluting.
    elif compound_class in ["Glycerophospholipid", "Sphingolipid"]:
        # Draw m/z values from a lipid-like mass range.
        mz = rng.uniform(450, 950)
        # Draw retention times from a later-eluting range.
        rt = rng.uniform(6.5, 14.5)
    # Nucleotides are mid-mass and relatively polar.
    elif compound_class == "Nucleotide":
        # Draw m/z values from a nucleotide-like mass range.
        mz = rng.uniform(250, 650)
        # Draw retention times from an early-to-mid range.
        rt = rng.uniform(1.5, 6.0)
    # Contaminants can span broad m/z and retention-time ranges.
    elif compound_class == "Xenobiotic / contaminant":
        # Draw m/z values from a broad range.
        mz = rng.uniform(120, 850)
        # Draw retention times from a broad LC gradient range.
        rt = rng.uniform(1.0, 15.0)
    # Unknown features can occur anywhere in the analytical space.
    else:
        # Draw m/z values from the full simulated range.
        mz = rng.uniform(70, 1000)
        # Draw retention times from the full simulated LC gradient.
        rt = rng.uniform(0.3, 15.0)
    # Return the sampled values.
    return mz, rt


# Loop over the simulated feature indices.
for feature_index in range(n_features):
    # Get the driver type for the current feature.
    driver_type = driver_types[feature_index]
    # Choose compound classes that match the driver type.
    if driver_type == "biology_stress_up":
        # Stress-associated biology is enriched for polar metabolites and acylcarnitines.
        compound_class = rng.choice(
            ["Amino acid / amine", "Organic acid", "Acylcarnitine"],
            p=[0.35, 0.40, 0.25],
        )
    elif driver_type == "biology_rescue_up":
        # Rescue-associated biology is enriched for lipid-like features.
        compound_class = rng.choice(
            ["Glycerophospholipid", "Sphingolipid", "Nucleotide"], p=[0.55, 0.25, 0.20]
        )
    elif driver_type == "lc_batch_shift":
        # Batch-sensitive features are often early-eluting or unknown features.
        compound_class = rng.choice(
            ["Organic acid", "Unknown", "Amino acid / amine"], p=[0.45, 0.35, 0.20]
        )
    elif driver_type == "injection_drift":
        # Drift can affect a broad range of features.
        compound_class = rng.choice(
            compound_classes, p=[0.10, 0.15, 0.10, 0.25, 0.10, 0.05, 0.05, 0.20]
        )
    elif driver_type == "extraction_protocol_shift":
        # Extraction effects often affect chemically coherent subsets of features.
        compound_class = rng.choice(
            ["Organic acid", "Glycerophospholipid", "Unknown"], p=[0.30, 0.45, 0.25]
        )
    elif driver_type == "blank_contaminant":
        # Blank-associated features are treated as xenobiotic or contaminant-like.
        compound_class = "Xenobiotic / contaminant"
    else:
        # Background features come from a broad mix of compound classes.
        compound_class = rng.choice(
            compound_classes, p=[0.12, 0.13, 0.08, 0.20, 0.10, 0.07, 0.05, 0.25]
        )
    # Sample an m/z and retention time consistent with the compound class.
    mz, rt = sample_mz_rt(compound_class, rng)
    # Choose an adduct label for the feature.
    adduct = rng.choice(adducts, p=[0.62, 0.20, 0.06, 0.08, 0.04])
    # Assign a lower annotation confidence to unknown and contaminant-like features.
    if compound_class in ["Unknown", "Xenobiotic / contaminant"]:
        # Unknowns and contaminants are more likely to remain unannotated.
        annotation_level = rng.choice(annotation_levels, p=[0.08, 0.32, 0.60])
    else:
        # Biochemically plausible classes are more likely to have tentative annotations.
        annotation_level = rng.choice(annotation_levels, p=[0.30, 0.45, 0.25])
    # Mark blank-associated features explicitly for later filtering exercises.
    blank_associated = driver_type == "blank_contaminant"
    # Create a feature identifier such as F_0001.
    feature_id = f"F_{feature_index + 1:04d}"
    # Store the complete feature metadata record.
    feature_records.append(
        {
            "feature_id": feature_id,
            "mz": mz,
            "rt_min": rt,
            "compound_class": compound_class,
            "adduct": adduct,
            "annotation_level": annotation_level,
            "driver_type": driver_type,
            "blank_associated": blank_associated,
        }
    )

# Convert the feature metadata list into a pandas DataFrame.
feature_metadata = pd.DataFrame(feature_records)

# Use feature identifiers as the DataFrame index.
feature_metadata = feature_metadata.set_index("feature_id")

# Show the first few rows of the feature metadata table.
feature_metadata.head()

In [ ]:
# Create an array of sample identifiers in the same order as sample_metadata.
sample_ids = sample_metadata.index.to_numpy()

# Create an array of feature identifiers in the same order as feature_metadata.
feature_ids = feature_metadata.index.to_numpy()

# Count the number of samples.
n_samples = len(sample_ids)

# Draw a baseline log2 intensity for each feature.
base_log2_intensity = rng.normal(loc=17.0, scale=1.7, size=n_features)

# Draw a sample-specific loading effect to mimic differences in total signal.
sample_loading_effect = rng.normal(loc=0.0, scale=0.25, size=n_samples)

# Standardize injection order to mean 0 and standard deviation 1 for drift simulation.
run_order_scaled = stats.zscore(sample_metadata["injection_order"].to_numpy())

# Create an empty matrix for simulated log2 intensities.
log2_matrix = np.zeros((n_samples, n_features))

# Loop over all samples.
for sample_index, sample_id in enumerate(sample_ids):
    # Retrieve sample metadata for the current sample.
    sample_row = sample_metadata.loc[sample_id]
    # Loop over all features.
    for feature_index, feature_id in enumerate(feature_ids):
        # Retrieve feature metadata for the current feature.
        feature_row = feature_metadata.loc[feature_id]
        # Start each simulated value at the feature-specific baseline intensity.
        value = base_log2_intensity[feature_index]
        # Add sample-specific total-signal variation to all non-blank samples.
        if sample_row["sample_type"] != "Blank":
            # Add a small multiplicative intensity effect on the log2 scale.
            value += sample_loading_effect[sample_index]
        # Simulate procedural blanks differently from real samples.
        if sample_row["sample_type"] == "Blank":
            # Make real endogenous metabolites nearly absent in blanks.
            value -= rng.uniform(5.0, 7.5)
            # Make blank contaminants abundant in blanks.
            if feature_row["driver_type"] == "blank_contaminant":
                # Increase contaminant features in blanks.
                value += rng.uniform(5.0, 6.5)
        # Simulate pooled QC samples as an approximate mixture of the biological samples.
        elif sample_row["sample_type"] == "QC":
            # Stress-up features are moderately present in pooled QC samples.
            if feature_row["driver_type"] == "biology_stress_up":
                # Add an average condition effect.
                value += 0.65
            # Rescue-up features are moderately present in pooled QC samples.
            if feature_row["driver_type"] == "biology_rescue_up":
                # Add an average condition effect.
                value += 0.35
            # Batch-sensitive features still respond to LC batch in QCs.
            if (
                feature_row["driver_type"] == "lc_batch_shift"
                and sample_row["lc_batch"] == "LC_Batch_2"
            ):
                # Add a batch shift to batch-sensitive features.
                value += 0.95
            # Drift-sensitive features still respond to injection order in QCs.
            if feature_row["driver_type"] == "injection_drift":
                # Add an injection-order drift effect.
                value += 0.65 * run_order_scaled[sample_index]
            # Blank-associated contaminants are usually weakly present in pooled QCs.
            if feature_row["driver_type"] == "blank_contaminant":
                # Add a small contaminant/carryover signal.
                value -= 0.25
        # Simulate biological experimental samples.
        else:
            # Stress-up features are most abundant in the Stress condition.
            if feature_row["driver_type"] == "biology_stress_up":
                # Define condition-specific effects on the log2 scale.
                condition_effect = {"Control": 0.00, "Stress": 1.55, "Rescue": 0.55}
                # Add the condition effect for this sample.
                value += condition_effect[sample_row["condition"]]
            # Rescue-up features are highest in the Rescue condition and lower in Stress.
            if feature_row["driver_type"] == "biology_rescue_up":
                # Define condition-specific effects on the log2 scale.
                condition_effect = {"Control": 0.15, "Stress": -0.45, "Rescue": 1.25}
                # Add the condition effect for this sample.
                value += condition_effect[sample_row["condition"]]
            # LC-batch features shift in the second half of the run.
            if (
                feature_row["driver_type"] == "lc_batch_shift"
                and sample_row["lc_batch"] == "LC_Batch_2"
            ):
                # Add an LC batch effect.
                value += 1.05
            # Drift features change gradually across injection order.
            if feature_row["driver_type"] == "injection_drift":
                # Add a smooth injection-order trend.
                value += 0.75 * run_order_scaled[sample_index]
            # Extraction-sensitive features shift between extraction protocols.
            if (
                feature_row["driver_type"] == "extraction_protocol_shift"
                and sample_row["extraction_protocol"] == "Protocol_B"
            ):
                # Add an extraction-protocol effect.
                value += 0.95
            # Blank-associated contaminant features are low but not completely absent in real samples.
            if feature_row["driver_type"] == "blank_contaminant":
                # Slightly reduce contaminant features in real samples relative to ordinary features.
                value -= 0.75
        # Add measurement noise, with QCs being slightly more reproducible.
        if sample_row["sample_type"] == "QC":
            # Add lower noise for QC samples.
            value += rng.normal(loc=0.0, scale=0.20)
        else:
            # Add ordinary analytical and biological noise.
            value += rng.normal(loc=0.0, scale=0.38)
        # Store the simulated log2 intensity in the matrix.
        log2_matrix[sample_index, feature_index] = value

# Convert log2 intensities to raw-like positive intensities.
raw_intensity_matrix = np.power(2.0, log2_matrix)

# Compute a dropout probability that is higher for low-abundance features.
dropout_probability = 1.0 / (1.0 + np.exp((log2_matrix - 12.0) / 1.4))

# Draw random dropout events from a uniform distribution.
dropout_events = rng.uniform(size=raw_intensity_matrix.shape) < dropout_probability

# Set dropped-out values to zero to mimic non-detected features.
raw_intensity_matrix[dropout_events] = 0.0

# Add a small amount of random missingness independent of abundance.
random_missing = rng.uniform(size=raw_intensity_matrix.shape) < 0.01

# Set randomly missing values to zero as well.
raw_intensity_matrix[random_missing] = 0.0

# Convert the raw intensity matrix into a pandas DataFrame.
feature_table = pd.DataFrame(
    raw_intensity_matrix, index=sample_ids, columns=feature_ids
)

# Show the shape of the simulated feature table.
feature_table.shape

In [ ]:
# Show the first five rows and first eight features of the feature table.
feature_table.iloc[:5, :8]

In [ ]:
# Count how many samples of each sample type are present.
sample_metadata["sample_type"].value_counts()

In [ ]:
# Count how many features belong to each simulated driver type.
feature_metadata["driver_type"].value_counts()

## Preprocessing helper functions

Dimensionality reduction is sensitive to preprocessing. In this exercise we use a common simple workflow:

1. optionally remove blanks or QCs;
2. optionally remove blank-associated features;
3. normalize each sample by its median non-zero intensity;
4. log-transform intensities;
5. impute remaining non-detected values with a small feature-specific value;
6. scale features before PCA, PCoA, or t-SNE.

This is not the only valid workflow. The point is to make preprocessing explicit so that you can reason about its effects.

In [ ]:
# Define a function that prepares the feature table for dimensionality reduction.
def preprocess_feature_table(
    feature_table,
    sample_metadata,
    feature_metadata,
    include_experimental=True,
    include_qc=True,
    include_blanks=True,
    remove_blank_associated_features=False,
    scaling="standard",
):
    # Start with all samples marked as excluded.
    sample_mask = pd.Series(False, index=sample_metadata.index)
    # Include experimental samples if requested.
    if include_experimental:
        # Mark experimental samples as included.
        sample_mask = sample_mask | (sample_metadata["sample_type"] == "Experimental")
    # Include QC samples if requested.
    if include_qc:
        # Mark QC samples as included.
        sample_mask = sample_mask | (sample_metadata["sample_type"] == "QC")
    # Include blank samples if requested.
    if include_blanks:
        # Mark blank samples as included.
        sample_mask = sample_mask | (sample_metadata["sample_type"] == "Blank")
    # Subset the feature table to the selected samples.
    X_raw = feature_table.loc[sample_mask].copy()
    # Subset the sample metadata to the selected samples.
    meta = sample_metadata.loc[sample_mask].copy()
    # Start with all features marked as included.
    feature_mask = pd.Series(True, index=feature_metadata.index)
    # Remove blank-associated features if requested.
    if remove_blank_associated_features:
        # Keep only features that are not marked as blank-associated.
        feature_mask = feature_mask & (~feature_metadata["blank_associated"])
    # Subset the feature table to the selected features.
    X_raw = X_raw.loc[:, feature_mask]
    # Subset the feature metadata to the selected features.
    feat_meta = feature_metadata.loc[feature_mask].copy()
    # Replace zeros with NaN so that they do not determine the sample median.
    X_nonzero = X_raw.replace(0, np.nan)
    # Compute each sample's median non-zero intensity.
    sample_medians = X_nonzero.median(axis=1)
    # Compute the global median across sample medians.
    global_median = sample_medians.median()
    # Normalize each sample to the global median intensity scale.
    X_norm = X_raw.div(sample_medians, axis=0) * global_median
    # Replace possible NaN values with zero after normalization.
    X_norm = X_norm.fillna(0.0)
    # Log2-transform the normalized intensities.
    X_log = np.log2(X_norm + 1.0)
    # Compute the smallest positive log-intensity for each feature.
    feature_min_positive = X_log.mask(X_log <= 0).min(axis=0)
    # Replace any all-zero feature minima with a conservative value of 1.0.
    feature_min_positive = feature_min_positive.fillna(1.0)
    # Define imputation values as half of the smallest positive log-intensity per feature.
    imputation_values = feature_min_positive / 2.0
    # Create a copy of the log-transformed data for imputation.
    X_log_imputed = X_log.copy()
    # Replace non-detected values with the feature-specific imputation value.
    X_log_imputed = X_log_imputed.mask(X_raw <= 0, imputation_values, axis=1)
    # Apply z-score scaling if requested.
    if scaling == "standard":
        # Create a StandardScaler object.
        scaler = StandardScaler()
        # Fit the scaler and transform the imputed log data.
        X_scaled_array = scaler.fit_transform(X_log_imputed)
        # Convert the scaled NumPy array back into a DataFrame.
        X_scaled = pd.DataFrame(
            X_scaled_array, index=X_log_imputed.index, columns=X_log_imputed.columns
        )
    # Apply Pareto scaling if requested.
    elif scaling == "pareto":
        # Compute feature means.
        feature_means = X_log_imputed.mean(axis=0)
        # Compute feature standard deviations.
        feature_stds = X_log_imputed.std(axis=0, ddof=1)
        # Replace zero standard deviations with one to avoid division by zero.
        feature_stds = feature_stds.replace(0, 1.0)
        # Center by the mean and divide by the square root of the standard deviation.
        X_scaled = (X_log_imputed - feature_means) / np.sqrt(feature_stds)
    # Use centered but unscaled log data if requested.
    elif scaling == "center_only":
        # Center each feature by subtracting its mean.
        X_scaled = X_log_imputed - X_log_imputed.mean(axis=0)
    # Raise an error for unknown scaling choices.
    else:
        # Stop execution with a clear error message.
        raise ValueError("scaling must be 'standard', 'pareto', or 'center_only'")
    # Return all useful intermediate objects.
    return {
        "X_raw": X_raw,
        "X_norm": X_norm,
        "X_log_imputed": X_log_imputed,
        "X_scaled": X_scaled,
        "sample_metadata": meta,
        "feature_metadata": feat_meta,
    }

In [ ]:
# Prepare a dataset that includes experimental samples, QCs, and blanks.
data_all = preprocess_feature_table(
    feature_table=feature_table,
    sample_metadata=sample_metadata,
    feature_metadata=feature_metadata,
    include_experimental=True,
    include_qc=True,
    include_blanks=True,
    remove_blank_associated_features=False,
    scaling="standard",
)

# Prepare a dataset containing only experimental biological samples.
data_experimental = preprocess_feature_table(
    feature_table=feature_table,
    sample_metadata=sample_metadata,
    feature_metadata=feature_metadata,
    include_experimental=True,
    include_qc=False,
    include_blanks=False,
    remove_blank_associated_features=False,
    scaling="standard",
)

# Prepare a cleaner experimental dataset with blank-associated features removed.
data_experimental_clean = preprocess_feature_table(
    feature_table=feature_table,
    sample_metadata=sample_metadata,
    feature_metadata=feature_metadata,
    include_experimental=True,
    include_qc=False,
    include_blanks=False,
    remove_blank_associated_features=True,
    scaling="standard",
)

# Print the shapes of the three prepared datasets.
print("All samples:", data_all["X_scaled"].shape)
print("Experimental only:", data_experimental["X_scaled"].shape)
print(
    "Experimental only, blank-associated features removed:",
    data_experimental_clean["X_scaled"].shape,
)

## PCA: what separates the samples?

PCA finds directions of maximum variance in the feature matrix. In MS applications, that variance can reflect biology, but it can also reflect blanks, batch, run order, total signal, extraction protocol, or contaminants.

In this section, first include **all** sample types. This is intentional: QC samples and blanks are part of quality assessment.

In [ ]:
# Define a function to run PCA on a prepared data dictionary.
def run_pca(preprocessed_data, n_components=5):
    # Extract the scaled feature matrix.
    X_scaled = preprocessed_data["X_scaled"]
    # Create a PCA model with the requested number of components.
    pca_model = PCA(
        n_components=n_components, svd_solver="randomized", random_state=RANDOM_SEED
    )
    # Fit PCA and compute sample scores.
    score_array = pca_model.fit_transform(X_scaled)
    # Create names for the principal components.
    pc_names = [f"PC{i + 1}" for i in range(n_components)]
    # Convert the sample scores into a DataFrame.
    scores = pd.DataFrame(score_array, index=X_scaled.index, columns=pc_names)
    # Attach sample metadata to the scores table.
    scores = scores.join(preprocessed_data["sample_metadata"])
    # Convert PCA loadings into a DataFrame with features as rows.
    loadings = pd.DataFrame(
        pca_model.components_.T, index=X_scaled.columns, columns=pc_names
    )
    # Attach feature metadata to the loadings table.
    loadings = loadings.join(preprocessed_data["feature_metadata"])
    # Return the fitted model, scores, and loadings.
    return pca_model, scores, loadings


# Run PCA on all samples, including QCs and blanks.
pca_all, pca_scores_all, pca_loadings_all = run_pca(data_all, n_components=5)

# Show the fraction of variance explained by the first five principal components.
pd.DataFrame(
    {
        "principal_component": [f"PC{i + 1}" for i in range(5)],
        "explained_variance_ratio": pca_all.explained_variance_ratio_,
    }
)

In [ ]:
# Define a reusable function for plotting two-dimensional embeddings.
def plot_embedding(scores, x_col, y_col, color_by, title):
    # Create a plotting area.
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    # Extract the metadata column used for coloring points.
    color_values = scores[color_by]
    # Check whether the metadata column is numeric.
    if pd.api.types.is_numeric_dtype(color_values):
        # Draw a scatter plot colored by the numeric values.
        scatter = ax.scatter(
            scores[x_col], scores[y_col], c=color_values, s=55, alpha=0.85
        )
        # Add a colorbar for the numeric metadata variable.
        fig.colorbar(scatter, ax=ax, label=color_by)
    # Handle categorical metadata columns.
    else:
        # Loop over the categories in sorted order.
        for category in sorted(color_values.astype(str).unique()):
            # Identify samples that belong to the current category.
            mask = color_values.astype(str) == category
            # Plot those samples as one category.
            ax.scatter(
                scores.loc[mask, x_col],
                scores.loc[mask, y_col],
                s=55,
                alpha=0.85,
                label=category,
            )
        # Add a legend outside the plotting area.
        ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", title=color_by)
    # Label the x-axis.
    ax.set_xlabel(x_col)
    # Label the y-axis.
    ax.set_ylabel(y_col)
    # Add the plot title.
    ax.set_title(title)
    # Use a tight layout to reduce clipping.
    fig.tight_layout()
    # Display the plot.
    plt.show()


# Choose which metadata variable to use for coloring.
COLOR_BY = "sample_type"

# Plot PC1 versus PC2 for all samples.
plot_embedding(
    scores=pca_scores_all,
    x_col="PC1",
    y_col="PC2",
    color_by=COLOR_BY,
    title=f"PCA of all samples colored by {COLOR_BY}",
)

### Hands-on task

Change `COLOR_BY` in the previous cell to each of the following:

- `condition`
- `lc_batch`
- `injection_order`
- `extraction_protocol`
- `column`

Discuss:

- What is the strongest separation when blanks and QCs are included?
- Where do the QC samples fall relative to the biological samples?
- Do blanks look like real biological samples?

In [ ]:
# Run PCA on experimental samples only.
pca_exp, pca_scores_exp, pca_loadings_exp = run_pca(data_experimental, n_components=5)

# Plot PC1 versus PC2 for experimental samples, colored by biological condition.
plot_embedding(
    scores=pca_scores_exp,
    x_col="PC1",
    y_col="PC2",
    color_by="condition",
    title="PCA of experimental samples colored by biological condition",
)

# Plot PC1 versus PC2 for experimental samples, colored by LC batch.
plot_embedding(
    scores=pca_scores_exp,
    x_col="PC1",
    y_col="PC2",
    color_by="lc_batch",
    title="PCA of experimental samples colored by LC batch",
)

# Plot PC1 versus PC2 for experimental samples, colored by injection order.
plot_embedding(
    scores=pca_scores_exp,
    x_col="PC1",
    y_col="PC2",
    color_by="injection_order",
    title="PCA of experimental samples colored by injection order",
)

In [ ]:
# Summarize how strongly each PC correlates with injection order.
for pc in ["PC1", "PC2", "PC3"]:
    # Compute the Spearman rank correlation between a PC and injection order.
    rho, p_value = stats.spearmanr(
        pca_scores_exp[pc], pca_scores_exp["injection_order"]
    )
    # Print the result in a readable format.
    print(f"{pc} vs injection_order: Spearman rho = {rho:.2f}, p = {p_value:.2e}")

### Interpretation prompt

For the experimental samples only:

- Which PC seems most biological?
- Which PC seems most technical?
- Is the technical effect stronger or weaker than the biological effect?
- How would your interpretation change if you only looked at one coloring variable?

## PCA loadings: which features drive the separation?

Scores tell us where samples are. Loadings tell us which features point in the same direction as a principal component.

In MS terms, loadings let us ask:

- Are high-loading features chemically plausible?
- Are they enriched for a compound class?
- Are they early-eluting features, lipids, contaminants, or unknowns?
- Are they blank-associated?
- Are they connected to sample prep or LC batch?

In [ ]:
# Define a helper function that returns the top positive and negative loading features.
def top_loadings(loadings, pc="PC1", n=12):
    # Sort features by the selected PC loading in descending order.
    positive = loadings.sort_values(pc, ascending=False).head(n).copy()
    # Sort features by the selected PC loading in ascending order.
    negative = loadings.sort_values(pc, ascending=True).head(n).copy()
    # Add a direction label to the positive-loading features.
    positive["loading_direction"] = "positive"
    # Add a direction label to the negative-loading features.
    negative["loading_direction"] = "negative"
    # Combine positive and negative loading features into one table.
    combined = pd.concat([positive, negative], axis=0)
    # Select columns that are useful for interpretation.
    columns_to_show = [
        pc,
        "loading_direction",
        "mz",
        "rt_min",
        "compound_class",
        "adduct",
        "annotation_level",
        "driver_type",
        "blank_associated",
    ]
    # Return the compact interpretation table.
    return combined[columns_to_show]


# Show the top features driving PC1 in the experimental-sample PCA.
top_loadings(pca_loadings_exp, pc="PC1", n=10)

In [ ]:
# Show the top features driving PC2 in the experimental-sample PCA.
top_loadings(pca_loadings_exp, pc="PC2", n=10)

In [ ]:
# Define a helper function to plot loadings in m/z-retention-time space.
def plot_loadings_mz_rt(loadings, pc="PC1", top_n=40):
    # Rank features by the absolute magnitude of their loading on the selected PC.
    ranked = loadings.reindex(
        loadings[pc].abs().sort_values(ascending=False).index
    ).head(top_n)
    # Create a plotting area.
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    # Draw a scatter plot of retention time versus m/z, colored by loading value.
    scatter = ax.scatter(ranked["rt_min"], ranked["mz"], c=ranked[pc], s=70, alpha=0.90)
    # Add a colorbar showing the loading value.
    fig.colorbar(scatter, ax=ax, label=f"{pc} loading")
    # Label the x-axis.
    ax.set_xlabel("Retention time (min)")
    # Label the y-axis.
    ax.set_ylabel("m/z")
    # Add a plot title.
    ax.set_title(f"Top {top_n} features by absolute {pc} loading")
    # Use a tight layout to reduce clipping.
    fig.tight_layout()
    # Display the plot.
    plt.show()


# Plot the top PC1 loading features in m/z-retention-time space.
plot_loadings_mz_rt(pca_loadings_exp, pc="PC1", top_n=50)

# Plot the top PC2 loading features in m/z-retention-time space.
plot_loadings_mz_rt(pca_loadings_exp, pc="PC2", top_n=50)

In [ ]:
# Define a helper function to plot feature intensity by sample group.
def plot_feature_by_group(feature_id, data, group_by="condition"):
    # Extract the normalized log-imputed intensities for one feature.
    values = data["X_log_imputed"][feature_id]
    # Extract the relevant sample metadata.
    meta = data["sample_metadata"]
    # Combine intensities and grouping metadata into one DataFrame.
    plot_df = pd.DataFrame({"intensity_log2": values, group_by: meta[group_by]})
    # Create a plotting area.
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    # Identify the groups to plot.
    groups = sorted(plot_df[group_by].astype(str).unique())
    # Loop over groups and plot each group separately.
    for group_index, group in enumerate(groups):
        # Select rows for the current group.
        group_values = plot_df.loc[
            plot_df[group_by].astype(str) == group, "intensity_log2"
        ]
        # Add small random horizontal jitter so points do not overlap perfectly.
        jitter = rng.normal(loc=0.0, scale=0.04, size=len(group_values))
        # Plot individual sample intensities.
        ax.scatter(
            np.repeat(group_index, len(group_values)) + jitter,
            group_values,
            alpha=0.80,
            s=45,
        )
        # Plot the group median as a larger marker.
        ax.scatter(group_index, group_values.median(), s=140, marker="_")
    # Label the x-axis ticks with group names.
    ax.set_xticks(range(len(groups)))
    # Rotate labels slightly for readability.
    ax.set_xticklabels(groups, rotation=20, ha="right")
    # Label the y-axis.
    ax.set_ylabel("log2 normalized intensity")
    # Add feature metadata to the plot title.
    feature_row = data["feature_metadata"].loc[feature_id]
    # Create a concise title for the feature.
    title = f"{feature_id}: m/z {feature_row['mz']:.4f}, RT {feature_row['rt_min']:.2f} min, {feature_row['compound_class']}"
    # Add the title to the plot.
    ax.set_title(title)
    # Use a tight layout to reduce clipping.
    fig.tight_layout()
    # Display the plot.
    plt.show()


# Select the strongest positive PC1 loading feature.
example_feature_pc1_positive = pca_loadings_exp.sort_values(
    "PC1", ascending=False
).index[0]

# Select the strongest negative PC1 loading feature.
example_feature_pc1_negative = pca_loadings_exp.sort_values(
    "PC1", ascending=True
).index[0]

# Plot the strongest positive PC1 loading feature by biological condition.
plot_feature_by_group(
    example_feature_pc1_positive, data_experimental, group_by="condition"
)

# Plot the strongest negative PC1 loading feature by biological condition.
plot_feature_by_group(
    example_feature_pc1_negative, data_experimental, group_by="condition"
)

### Hands-on task

Choose one feature from the PC1 or PC2 loading tables and replace the feature ID below.

Questions:

- Does the feature differ by biological condition, LC batch, injection order, or extraction protocol?
- Is its compound class chemically plausible for that interpretation?
- Would you trust this feature as biological evidence, or is it more likely technical?

In [ ]:
# Choose a feature ID to investigate.
FEATURE_TO_INSPECT = example_feature_pc1_positive

# Plot the chosen feature by biological condition.
plot_feature_by_group(FEATURE_TO_INSPECT, data_experimental, group_by="condition")

# Plot the same feature by LC batch.
plot_feature_by_group(FEATURE_TO_INSPECT, data_experimental, group_by="lc_batch")

# Plot the same feature by extraction protocol.
plot_feature_by_group(
    FEATURE_TO_INSPECT, data_experimental, group_by="extraction_protocol"
)

In [ ]:
# Count the driver types among the top absolute loading features for PC1.
pca_loadings_exp.reindex(
    pca_loadings_exp["PC1"].abs().sort_values(ascending=False).head(50).index
)["driver_type"].value_counts()

In [ ]:
# Count the driver types among the top absolute loading features for PC2.
pca_loadings_exp.reindex(
    pca_loadings_exp["PC2"].abs().sort_values(ascending=False).head(50).index
)["driver_type"].value_counts()

### Intermediate challenge

Rerun PCA after removing blank-associated features. Does this change the top loadings or the apparent sample separation?

In [ ]:
# Run PCA on experimental samples after removing blank-associated features.
pca_exp_clean, pca_scores_exp_clean, pca_loadings_exp_clean = run_pca(
    data_experimental_clean, n_components=5
)

# Plot PC1 versus PC2 for the cleaned experimental dataset.
plot_embedding(
    scores=pca_scores_exp_clean,
    x_col="PC1",
    y_col="PC2",
    color_by="condition",
    title="PCA after removing blank-associated features",
)

# Show the top features driving PC1 after blank-associated features are removed.
top_loadings(pca_loadings_exp_clean, pc="PC1", n=10)

## PCoA: what changes when we choose a distance metric?

PCA operates directly on the feature matrix. PCoA starts from a sample-by-sample distance matrix.

This distinction matters in MS data because the distance metric encodes a scientific assumption. For example:

- **Euclidean distance** emphasizes absolute differences after scaling.
- **Correlation distance** emphasizes similarity in feature patterns, regardless of absolute scale.
- **Bray-Curtis distance** is often used for compositional abundance profiles and is sensitive to shared versus non-shared signal.

In this section, we compare these distance metrics using the same experimental samples.

In [ ]:
# Define a function that runs classical MDS / PCoA directly from a data matrix.
def run_pcoa(data, metric="euclidean", n_components=2):
    """
    Run classical MDS / PCoA using a named distance metric.

    Parameters
    ----------
    data : dict
        Dictionary produced by the preprocessing function.
        It should contain:
        - data["X_scaled"]: centered/scaled feature matrix.
        - data["X_norm"]: median-normalized non-negative intensity matrix.
        - data["sample_ids"]: sample identifiers.

    metric : str
        Distance metric to use.
        Recommended options for this notebook:
        - "euclidean": PCA-like geometry when applied to the same scaled matrix.
        - "correlation": compares sample profiles rather than absolute abundance.
        - "braycurtis": compares compositional abundance differences.

    n_components : int
        Number of ordination dimensions to return.

    Returns
    -------
    coordinates_df : pandas.DataFrame
        Sample coordinates in the low-dimensional PCoA space.

    explained : numpy.ndarray
        Fraction of positive classical-MDS eigenvalue mass represented by each
        displayed axis.
    """

    # Check that the requested metric is one of the metrics used in this notebook.
    if metric not in ["euclidean", "correlation", "braycurtis"]:
        # Stop execution with a clear message if the user typed an unsupported metric.
        raise ValueError("metric must be 'euclidean', 'correlation', or 'braycurtis'")

    # For Euclidean and correlation distances, use the scaled feature matrix.
    # This makes Euclidean classical MDS directly comparable to PCA.
    if metric in ["euclidean", "correlation"]:
        # Extract the scaled sample-by-feature matrix.
        X = data["X_scaled"]

    # For Bray-Curtis distance, use non-negative normalized intensities.
    # Bray-Curtis is intended for non-negative abundance-like data.
    elif metric == "braycurtis":
        # Extract the median-normalized intensity matrix.
        X = data["X_norm"]

    # Convert the sample identifiers to a list.
    sample_ids = list(data["X_scaled"].index)

    # Check that there is exactly one sample identifier per row of the data matrix.
    if len(sample_ids) != X.shape[0]:
        # Stop execution with a clear error message if the identifiers do not match.
        raise ValueError(
            "sample_ids must contain one identifier per row of the data matrix"
        )

    # ClassicalMDS only returns eigenvalues for the components it computes.
    # To estimate axis fractions, compute as many possible axes as there are samples minus one.
    n_full_components = min(X.shape[0] - 1, X.shape[1])

    # Create the classical MDS model.
    pcoa = ClassicalMDS(n_components=n_full_components, metric=metric)

    # Fit classical MDS and compute the ordination coordinates.
    full_coordinates = pcoa.fit_transform(X)

    # Keep only the requested number of displayed components.
    coordinates = full_coordinates[:, :n_components]

    # Create coordinate column names.
    coordinate_names = [f"PCoA{i + 1}" for i in range(n_components)]

    # Convert the coordinates to a DataFrame indexed by sample identifier.
    coordinates_df = pd.DataFrame(
        coordinates, index=sample_ids, columns=coordinate_names
    )

    # Extract the classical MDS eigenvalues.
    eigenvalues = np.asarray(pcoa.eigenvalues_)

    # Keep only positive eigenvalues when computing explained fractions.
    positive_total = eigenvalues[eigenvalues > 0].sum()

    # Compute the fraction of positive eigenvalue mass for the displayed axes.
    explained = np.maximum(eigenvalues[:n_components], 0) / positive_total

    # Return the coordinates and axis fractions.
    return coordinates_df, explained

In [ ]:
# Choose the distance metric for PCoA.
DISTANCE_METRIC = "euclidean"

# Run PCoA on the selected distance matrix.
# Choose the distance metric for PCoA.
pcoa_metric = "euclidean"  # Try: "euclidean", "correlation", or "braycurtis"

# Run classical MDS / PCoA.
pcoa_scores, pcoa_explained = run_pcoa(
    data=data_experimental, metric=DISTANCE_METRIC, n_components=5
)

# Attach sample metadata to the PCoA coordinates.
pcoa_scores = pcoa_scores.join(data_experimental["sample_metadata"])

# Print the variance explained by the first two PCoA axes.
print(f"{DISTANCE_METRIC} PCoA axis 1 explained fraction: {pcoa_explained[0]:.3f}")
print(f"{DISTANCE_METRIC} PCoA axis 2 explained fraction: {pcoa_explained[1]:.3f}")

# Plot PCoA axis 1 versus axis 2 colored by biological condition.
plot_embedding(
    scores=pcoa_scores,
    x_col="PCoA1",
    y_col="PCoA2",
    color_by="condition",
    title=f"PCoA using {DISTANCE_METRIC} distance, colored by condition",
)

### Hands-on task

Change `DISTANCE_METRIC` in the previous cell to:

- `euclidean`
- `correlation`
- `braycurtis`

Then also change `color_by` in the plotting call to:

- `condition`
- `lc_batch`
- `injection_order`
- `extraction_protocol`

Questions:

- Which distance metric best separates biological condition?
- Which distance metric makes technical structure most visible?
- Does changing the distance metric change the scientific conclusion?

In [ ]:
# Define a helper function to calculate simple embedding diagnostics.
def embedding_diagnostics(scores, x_col, y_col, label_col):
    # Extract the two-dimensional coordinates.
    coordinates = scores[[x_col, y_col]].to_numpy()
    # Extract group labels.
    labels = scores[label_col].astype(str).to_numpy()
    # Check whether at least two groups are present.
    if len(np.unique(labels)) < 2:
        # Return missing value if the diagnostic is not defined.
        return np.nan
    # Compute the silhouette score for the chosen labels.
    return silhouette_score(coordinates, labels)


# Create an empty list to collect PCoA diagnostics.
pcoa_diagnostic_records = []

# Loop over the three distance metrics.
for metric in ["euclidean", "correlation", "braycurtis"]:
    # Run PCoA on the current distance matrix.
    coords_metric, explained_metric = run_pcoa(
        data=data_experimental, metric=metric, n_components=5
    )
    # Attach sample metadata.
    coords_metric = coords_metric.join(data_experimental["sample_metadata"])
    # Compute the condition silhouette score.
    condition_silhouette = embedding_diagnostics(
        coords_metric, "PCoA1", "PCoA2", "condition"
    )
    # Compute the LC-batch silhouette score.
    batch_silhouette = embedding_diagnostics(
        coords_metric, "PCoA1", "PCoA2", "lc_batch"
    )
    # Compute the Spearman correlation between PCoA1 and injection order.
    run_rho, run_p = stats.spearmanr(
        coords_metric["PCoA1"], coords_metric["injection_order"]
    )
    # Store the diagnostic results.
    pcoa_diagnostic_records.append(
        {
            "metric": metric,
            "axis1_explained": explained_metric[0],
            "axis2_explained": explained_metric[1],
            "silhouette_condition": condition_silhouette,
            "silhouette_lc_batch": batch_silhouette,
            "spearman_axis1_injection_order": run_rho,
        }
    )

# Convert diagnostics to a DataFrame.
pcoa_diagnostics = pd.DataFrame(pcoa_diagnostic_records)

# Show the diagnostic table.
pcoa_diagnostics

### Interpretation prompt

A higher silhouette score means that samples with the same label are more compactly separated in the displayed two-dimensional embedding. It is not a proof of biology; it is just a useful summary.

Use the diagnostic table to discuss:

- Which metric emphasizes condition?
- Which metric emphasizes LC batch?
- Does PCoA axis 1 appear to track injection order for any metric?
- Why might an MS analyst choose one distance metric over another?

## Optional intermediate extensions

### A. Scaling sensitivity

Change `scaling` in `preprocess_feature_table` from `standard` to `pareto` or `center_only`, rerun PCA, and inspect how the scores and loadings change.

### B. Feature-space dimensionality reduction

Transpose the matrix so that each point is a feature rather than a sample. Then ask whether features cluster by compound class, m/z, retention time, blank association, or driver type.

### C. Remove technical features

Filter out features marked as `lc_batch_shift`, `injection_drift`, or `blank_contaminant`, rerun PCA, and ask what biological structure remains.

In [ ]:
# Extension B starter code: feature-space PCA.
# Here, rows are features and columns are samples.
X_feature_space = data_experimental_clean["X_scaled"].T

# Run PCA where features are the observations.
pca_feature_space = PCA(
    n_components=2, svd_solver="randomized", random_state=RANDOM_SEED
)

# Compute feature-space PCA coordinates.
feature_space_array = pca_feature_space.fit_transform(X_feature_space)

# Convert the coordinates into a DataFrame.
feature_space_scores = pd.DataFrame(
    feature_space_array, index=X_feature_space.index, columns=["Dim1", "Dim2"]
)

# Attach feature metadata so that features can be colored or interpreted.
feature_space_scores = feature_space_scores.join(
    data_experimental_clean["feature_metadata"]
)

# Plot feature-space PCA colored by compound class.
plot_embedding(
    scores=feature_space_scores,
    x_col="Dim1",
    y_col="Dim2",
    color_by="compound_class",
    title="Feature-space PCA: each point is an LC-MS feature",
)

# Bonus: Explore t-SNE hyperparameters

The performance of t-SNE can be heavily influenced by its hyperparameters. Additionally, because t-SNE uses a non-deterministic optimization procedure, its results can differ even with the same hyperparameters (unlike for PCA). Suboptimal t-SNE hyperparameters might suggest a data clustering that is not present in the full-dimensional data.

The Distill article ["How to use t-SNE effectively?"](https://distill.pub/2016/misread-tsne/) allows you to interactively explore the effect of different t-SNE hyperparameters on several toy datasets. (The hyperparameter "epsilon" is the same as the "learning rate" in the lecture.)

**Play around with t-SNE hyperparameters.**

- Select different datasets and hyperparameter combinations. Let t-SNE run until convergence.
- Re-run t-SNE with unchanged hyperparameters. Is the embedding identical?
- Did you find typical failure cases of t-SNE? What is the influence of the perplexity hyperparameter?